# OpenAlex author through the papers (DOI match): is it the same person?

`scripts/legacy/2_doi_match.py` (replaced by `scripts/5_openalex_cascade/2_jw_match.py`) picks, for each Semantic Scholar researcher with `factuality_author = True`, the OpenAlex author with the same last name who signs most of the researcher's papers (by DOI). Here it is checked with evidence the match does not use: the first name, the first-publication year, and how many of the researcher's papers that author signs. Sample: the researchers downloaded so far (`doi_match_sample.parquet`).

In [1]:
import sys
import pandas as pd

sys.path.insert(0, "../..")
sys.path.insert(0, "../../scripts/6_enrichment")
from oa_common import normalize_name
from libs.utils.config import config_default

R = "../../../results/summary"
LEGACY = "../../../results/legacy/openalex_cascade"
match = pd.read_parquet(f"{LEGACY}/doi_match_sample.parquet")
ss = (pd.read_parquet(config_default("ss_parquet"), columns=["Researcher_id", "Name", "First_year", "Field"])
        .drop_duplicates("Researcher_id").rename(columns={"Researcher_id": "researcher_id"}))
ss["researcher_id"] = ss["researcher_id"].astype("int64")
df = match.merge(ss, on="researcher_id", how="left")
len(df)

6456

## 1. First name
The match only checks the last name. Does the first name agree too (the same, or the same initial when one side only has an initial)?

In [2]:
def first_name(ss_name, oa_name):
    a, b = normalize_name(ss_name).split(), normalize_name(oa_name).split()
    if not a or not b:
        return "unknown"
    if a[0] == b[0]:
        return "same first name"
    if a[0][0] == b[0][0] and (len(a[0]) == 1 or len(b[0]) == 1):
        return "same initial"
    return "different"

df["first_name"] = [first_name(a, b) for a, b in zip(df["Name"], df["display_name"])]
df["first_name"].value_counts(normalize=True).round(3)

first_name
same first name    0.931
different          0.036
same initial       0.033
unknown            0.000
Name: proportion, dtype: float64

## 2. First-publication year
Semantic Scholar only sees papers of 6 fields, so its first year is usually later than OpenAlex's. The same person should not have an OpenAlex first year much *later* than Semantic Scholar's.

In [3]:
df["year_gap"] = df["First_year"] - df["first_year"]   # > 0: OpenAlex starts earlier
pd.Series({
    "OpenAlex starts earlier or the same year": (df["year_gap"] >= 0).mean(),
    "OpenAlex starts 1–5 years later": df["year_gap"].between(-5, -1).mean(),
    "OpenAlex starts > 5 years later": (df["year_gap"] < -5).mean(),
    "median gap (years)": df["year_gap"].median(),
}).round(3)

OpenAlex starts earlier or the same year    0.917
OpenAlex starts 1–5 years later             0.039
OpenAlex starts > 5 years later             0.044
median gap (years)                          9.000
dtype: float64

## 3. How much of the researcher's work the OpenAlex author signs
`n_shared / n_dois`: share of the researcher's papers found in OpenAlex that are signed by the matched author.

In [4]:
df["share"] = df["n_shared"] / df["n_dois"]
pd.DataFrame({
    "researchers": [len(df)],
    "median papers in common": [df["n_shared"].median()],
    "median share signed": [df["share"].median()],
    "share >= 0.8": [(df["share"] >= 0.8).mean()],
    "share < 0.5": [(df["share"] < 0.5).mean()],
    "only 1 paper in common": [(df["n_shared"] == 1).mean()],
}).round(3)

,researchers,median papers in common,median share signed,share >= 0.8,share < 0.5,only 1 paper in common
0,6456,26.0,0.95,0.802,0.036,0.074


## 4. All three together

In [5]:
ok_name = df["first_name"].isin(["same first name", "same initial"])
ok_year = df["year_gap"] >= -5
ok_share = df["share"] >= 0.5
pd.Series({
    "first name + year + share": (ok_name & ok_year & ok_share).mean(),
    "fails the first name": (~ok_name).mean(),
    "fails the year": (~ok_year).mean(),
    "fails the share": (~ok_share).mean(),
}).round(3)

first name + year + share    0.900
fails the first name         0.036
fails the year               0.044
fails the share              0.036
dtype: float64

## 5. Against the profile the cascade picked
Per recommendation with `factuality_author = True`: is the DOI profile the cascade's one? When they differ, which of the two agrees on the first name?

In [6]:
rec = pd.read_csv(f"{R}/factuality_full.csv", usecols=["researcher_id", "factuality_author", "oa_id", "oa_display_name"],
                  low_memory=False)
rec = rec[rec["factuality_author"]].dropna(subset=["researcher_id"])
rec["researcher_id"] = rec["researcher_id"].astype("int64")
cmp = rec.merge(df[["researcher_id", "oa_id", "display_name", "Name", "first_name"]], on="researcher_id",
                suffixes=("_cascade", "_doi"))
cmp["same"] = cmp["oa_id_cascade"] == cmp["oa_id_doi"]
cmp["cascade_first_name"] = [first_name(a, b) if isinstance(b, str) else "no profile"
                             for a, b in zip(cmp["Name"], cmp["oa_display_name"])]
diff = cmp[~cmp["same"]]
pd.Series({
    "recommendations": len(cmp),
    "same profile": cmp["same"].mean(),
    "cascade had no profile": cmp["oa_id_cascade"].isna().mean(),
    "different: DOI first name ok": diff["first_name"].isin(["same first name", "same initial"]).mean(),
    "different: cascade first name ok": diff["cascade_first_name"].isin(["same first name", "same initial"]).mean(),
}).round(3)

recommendations                     46174.000
same profile                            0.423
cascade had no profile                  0.219
different: DOI first name ok            0.958
different: cascade first name ok        0.516
dtype: float64